In [ ]:
from fastai.vision.all import *
from torch.nn import *

torch.cuda.set_device(1)


class Dat:
    a = torch.from_numpy(np.memmap(f"../tmp/news2016zh.H", "h"))
    m = np.array([open("../tmp/news2016zh.txt").read()]).view("U1")

    def __init__(s, w, b, n=1):
        s.a = s.a.as_strided([len(s.a) - w + 1, w], [1, 1])
        s.b = b
        s.n = n

    def __len__(s):
        return len(s.a) // (s.a.size(1) * s.b * s.n)

    def __iter__(s):
        for x in range(len(s)):
            x = s.a[torch.randint(len(s.a), [s.b])].to("cuda", torch.long)
            yield x, x


class View(Module):
    def __init__(s, k):
        super().__init__()
        s.k = k

    def forward(s, x):
        return x.view(*x.shape[:-2], -1, s.k)


class PyramidAutoEncoder(Sequential):
    def forward(s, x):
        n = int(np.log2(x.size(1)))
        t = []
        for _ in range(n):
            x = s[0](x)
            t.append(x)
        t = list(s[1](torch.cat(t, 1)).split([t.size(1) for t in t], 1))
        x = None
        while t:
            x = s[2](t.pop()[None] if x is None else torch.cat([t.pop()[None], x]))
        return x


class MultiCrossEntropyLoss(CrossEntropyLoss):
    def forward(s, x, t):
        x = x.view(len(x), -1, x.size(-1))
        return -x.log_softmax(2)[:, torch.arange(x.size(1), device="cuda"), t.flatten()
        ].mean()


def Accuracy(i):
    def Accuracy(x, t):
        return accuracy(x[i], t)

    Accuracy.__name__ = str(int(i))
    return Accuracy

In [ ]:
m = Learner(
    DataLoaders(Dat(256, 8, 500), Dat(256, 8, 50000)),
    Sequential(
        Embedding(21482, 256),
        PyramidAutoEncoder(
            Sequential(View(512), Linear(512, 256, 0), BatchNorm1dFlat(256, track_running_stats=0), Swish()),
            Sequential(Linear(256, 256, 0), BatchNorm1dFlat(256, track_running_stats=0), Swish(), Linear(256, 256)),
            Sequential(BatchNorm1dFlat(256, track_running_stats=0), Swish(), Linear(256, 512, 0), View(256)),
        ),
        Linear(256, 256, 0),
        BatchNorm1dFlat(256, track_running_stats=0),
        Swish(),
        Linear(256, 21482),
    ).cuda(),
    MultiCrossEntropyLoss(),
    metrics=map(Accuracy, range(8)),
    cbs=[ShowGraphCallback, SaveModelCallback(fname='modelb', every_epoch=1, with_opt=1, reset_on_fit=0)],
)
m.load('model', 'cuda')
# m.lr_find()

In [ ]:
x = next(iter(m.dls[1]))[0]
print(m.dls.m[x[0].cpu()].view('U256')[0])

In [ ]:
for i in m.dls.m[m.model(x)[:,0].argmax(2).cpu()].view('U256')[:,0]:
    print(i, end='\n====================\n')